# Chicago Weather Fetcher Playground

Use this notebook to explore the NOAA weather fetcher components in `core.weather_fetcher`.

In [11]:
# core lives in the parent directory, so we need to add it to the path
# This notebook will fetch weather data for Chicago and visualize it.
import sys
import os
import geopandas as gpd
from shapely.geometry import Point

# Get parent directory (project root)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add to Python path
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


from core import weather_fetcher, geo_utils, config
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd


In [6]:
# Rough Chicago viewport (you can tweak these)
CHICAGO_CENTER = {"lat": 41.8781, "lon": -87.6298}
CHICAGO_BBOX = geo_utils.BoundingBox(
    lat_min=41.60,
    lat_max=42.10,
    lon_min=-88.00,
    lon_max=-87.40,
 )

print("Weather API:", config.WEATHER_BASE_URL)
print("Station limit:", config.WEATHER_STATION_LIMIT)
print("Point sample limit:", config.WEATHER_POINT_SAMPLE_LIMIT)

Weather API: https://api.weather.gov
Station limit: 10
Point sample limit: 5


In [7]:
# 1) Pure geometry component: sampled points used to discover nearby NOAA station lists
sample_points = weather_fetcher._sample_points(CHICAGO_BBOX, CHICAGO_CENTER)
sample_points

[(41.8781, -87.6298),
 (41.6, -88.0),
 (41.6, -87.4),
 (42.1, -88.0),
 (42.1, -87.4)]

In [8]:
# 2) Network component: candidate stations near sampled points
candidates = weather_fetcher._station_candidates(CHICAGO_BBOX, CHICAGO_CENTER)
candidates_df = pd.DataFrame(candidates)
candidates_df.head(10)

,station_id,name,latitude,longitude,distance_to_center_km
0,KMDW,"Chicago, Chicago Midway Airport",41.78417,-87.75528,14.736789
1,KORD,"Chicago, Chicago-O'Hare International Airport",41.97972,-87.90444,25.374701
2,KGYY,Gary Regional Airport,41.61212,-87.40908,34.785614
3,KPWK,"Chicago / Wheeling, Pal-Waukee Airport",42.12083,-87.90472,35.278593
4,KIGQ,Lansing Municipal Airport,41.54125,-87.52822,38.393399
5,KLOT,Lewis University Airport,41.60307,-88.10164,49.677459
6,KDPA,"Chicago / West Chicago, Dupage Airport",41.89639,-88.25111,51.472245
7,KJOT,Joliet Regional Airport,41.51755,-88.17903,60.717687
8,KUGN,Chicago/Waukegan Regional Airport,42.42546,-87.86339,63.837149
9,KVPZ,Valparaiso Porter County Municipal Airport,41.45349,-86.99805,70.590484


In [9]:
# 3) End-to-end component: fetch + normalize latest observations
overlay = weather_fetcher.fetch_workspace_weather(CHICAGO_BBOX, CHICAGO_CENTER)

print("Available:", overlay.get("available"))
print("Summary:")
for line in overlay.get("summary", []):
    print(" -", line)

weather_df = pd.DataFrame(overlay.get("points", []))
weather_df.head(10)

Available: True
Summary:
 - Weather stations loaded: 10
 - Request failures: 0
 - Latest observation: 2026-03-21T11:00:00+00:00
 - Cloudy / obstructed stations: 3
 - Mean wind speed: 12.1 mph
 - Mean temperature: 42.2 °F


,station_id,name,latitude,longitude,distance_to_center_km,timestamp,temperature_c,temperature_f,wind_speed_m_s,wind_speed_mph,wind_direction_deg,wind_direction_cardinal,pressure_pa,pressure_hpa,cloud_amounts,has_clouds,text_description
0,KMDW,"Chicago, Chicago Midway Airport",41.78417,-87.75528,14.736789,2026-03-21T11:00:00+00:00,7.0,44.60,7.416,16.589147,120.0,SE,101354.61,1013.5461,[],False,
1,KORD,"Chicago, Chicago-O'Hare International Airport",41.97972,-87.90444,25.374701,2026-03-21T11:00:00+00:00,6.0,42.80,7.416,16.589147,110.0,E,101320.75,1013.2075,[],False,
2,KGYY,Gary Regional Airport,41.61212,-87.40908,34.785614,2026-03-21T09:15:00+00:00,5.0,41.00,0.000,0.000000,0.0,N,101390.00,1013.9000,[OVC],True,Cloudy
3,KPWK,"Chicago / Wheeling, Pal-Waukee Airport",42.12083,-87.90472,35.278593,2026-03-21T11:00:00+00:00,4.0,39.20,NaN,NaN,NaN,NaN,101388.48,1013.8848,[CLR],False,Clear
4,KIGQ,Lansing Municipal Airport,41.54125,-87.52822,38.393399,2026-03-21T10:55:00+00:00,5.2,41.36,7.560,16.911266,90.0,E,101390.00,1013.9000,[CLR],False,Clear
5,KLOT,Lewis University Airport,41.60307,-88.10164,49.677459,2026-03-21T10:45:00+00:00,7.0,44.60,7.560,16.911266,80.0,E,101390.00,1013.9000,[SCT],True,Partly Cloudy
6,KDPA,"Chicago / West Chicago, Dupage Airport",41.89639,-88.25111,51.472245,2026-03-21T10:50:00+00:00,5.0,41.00,5.544,12.401595,90.0,E,101320.75,1013.2075,[CLR],True,Fog/Mist
7,KJOT,Joliet Regional Airport,41.51755,-88.17903,60.717687,2026-03-21T10:55:00+00:00,8.3,46.94,7.560,16.911266,100.0,E,101390.00,1013.9000,[CLR],False,Clear
8,KUGN,Chicago/Waukegan Regional Airport,42.42546,-87.86339,63.837149,2026-03-21T11:00:00+00:00,3.0,37.40,5.544,12.401595,70.0,E,101354.61,1013.5461,[CLR],False,Clear
9,KVPZ,Valparaiso Porter County Municipal Airport,41.45349,-86.99805,70.590484,2026-03-21T11:00:00+00:00,6.0,42.80,0.000,0.000000,0.0,N,101456.20,1014.5620,[CLR],False,Clear


In [10]:
# 4) Playground helper: quickly rerun with custom bounding boxes
def run_weather_playground(lat_min=41.60, lat_max=42.10, lon_min=-88.00, lon_max=-87.40, center_lat=41.8781, center_lon=-87.6298):
    bbox = geo_utils.BoundingBox(lat_min=lat_min, lat_max=lat_max, lon_min=lon_min, lon_max=lon_max)
    center = {"lat": center_lat, "lon": center_lon}
    data = weather_fetcher.fetch_workspace_weather(bbox, center)
    df = pd.DataFrame(data.get("points", []))
    return data.get("summary", []), df

summary, df = run_weather_playground()
summary, df.head(5)

(['Weather stations loaded: 10',
  'Request failures: 0',
  'Latest observation: 2026-03-21T11:00:00+00:00',
  'Cloudy / obstructed stations: 3',
  'Mean wind speed: 12.1 mph',
  'Mean temperature: 42.2 °F'],
   station_id                                           name  latitude  \
 0       KMDW                Chicago, Chicago Midway Airport  41.78417   
 1       KORD  Chicago, Chicago-O'Hare International Airport  41.97972   
 2       KGYY                          Gary Regional Airport  41.61212   
 3       KPWK         Chicago / Wheeling, Pal-Waukee Airport  42.12083   
 4       KIGQ                      Lansing Municipal Airport  41.54125   
 
    longitude  distance_to_center_km                  timestamp  temperature_c  \
 0  -87.75528              14.736789  2026-03-21T11:00:00+00:00            7.0   
 1  -87.90444              25.374701  2026-03-21T11:00:00+00:00            6.0   
 2  -87.40908              34.785614  2026-03-21T09:15:00+00:00            5.0   
 3  -87.90472    

In [ ]:
# map weather stations
# Convert station coordinates to GeoDataFrame
geometry = [Point(lon, lat) for lon, lat in zip(candidates_df['longitude'], candidates_df['latitude'])]
gdf = gpd.GeoDataFrame(candidates_df, geometry=geometry)
# Plot stations on a map using folium
import folium
from folium.plugins import MarkerCluster
# Create a folium map centered on Chicago
m = folium.Map(location=[CHICAGO_CENTER['lat'], CHICAGO_CENTER['lon']], zoom_start=10)
# Add station markers to the map
marker_cluster = MarkerCluster().add_to(m)
for idx, row in gdf.iterrows():
    folium.Marker(location=[row['latitude'], row['longitude']], popup=row['name']).add_to(marker_cluster)
# Display the map
m.save("chicago_weather_stations.html")
display(m)   